In [ ]:
def generate_sign(path, partner_id, partner_key, timestamp, access_token=None, shop_id=None):
    """Generates the HMAC-SHA256 signature required by Shopee."""
    
    # Start with the base components required by ALL Shopee APIs
    base_string = f"{partner_id}{path}{timestamp}"
    
    # Append token and shop_id if this is a Shop-Level API
    if access_token and shop_id:
        base_string += f"{access_token}{shop_id}"
        
    return hmac.new(
        partner_key.encode(), 
        base_string.encode(), 
        hashlib.sha256
    ).hexdigest()

In [ ]:
def get_shop_info(partner_id, partner_key, shop_id, access_token):
    host = "https://partner.shopeemobile.com"
    path = "/api/v2/shop/get_shop_info"
    timestamp = int(time.time())
    
    # 1. Generate the signature cleanly
    sign = generate_sign(path, partner_id, partner_key, timestamp, access_token, shop_id)
    
    # 2. Build the URL
    url = (f"{host}{path}?partner_id={partner_id}&timestamp={timestamp}"
           f"&access_token={access_token}&shop_id={shop_id}&sign={sign}")
    
    # 3. Execute
    resp = requests.get(url)
    return resp.json()

In [ ]:
def make_shopee_request(path, partner_id, partner_key, shop_id, access_token, additional_params=None):
    """Handles the boilerplate of making an authenticated Shopee API GET request."""
    host = "https://partner.shopeemobile.com"
    timestamp = int(time.time())
    sign = generate_sign(path, partner_id, partner_key, timestamp, access_token, shop_id)
    
    params = {
        "partner_id": partner_id,
        "timestamp": timestamp,
        "access_token": access_token,
        "shop_id": shop_id,
        "sign": sign
    }
    
    if additional_params:
        params.update(additional_params)
        
    response = requests.get(host + path, params=params)
    data = response.json()
    
    if data.get("error"):
        print(f"API Error [{path}]: {data.get('message')}")
        return None
        
    return data.get("response")

def save_to_files(data_list, filename_prefix):
    """Saves a list of dictionaries to both a JSON and a CSV file."""
    if not data_list:
        print(f"No data to save for {filename_prefix}.")
        return

    os.makedirs("data_exports", exist_ok=True)
    json_path = f"data_exports/{filename_prefix}.json"
    csv_path = f"data_exports/{filename_prefix}.csv"

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, indent=4, ensure_ascii=False)

    all_keys = []
    for row in data_list:
        for key in row.keys():
            if key not in all_keys:
                all_keys.append(key)
                
    with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
        dict_writer = csv.DictWriter(f, fieldnames=all_keys)
        dict_writer.writeheader()
        
        for row in data_list:
            clean_row = {k: (str(v) if isinstance(v, (dict, list)) else v) for k, v in row.items()}
            dict_writer.writerow(clean_row)
            
    print(f"Saved {len(data_list)} records to {json_path} and {csv_path}")

In [ ]:
def get_all_item_ids(partner_id, partner_key, shop_id, access_token):
    """Fetches the Master List of all NORMAL status item IDs in the shop."""
    print("Fetching Master Item List...")
    item_ids = []
    offset = 0
    has_next = True
    
    while has_next:
        data = make_shopee_request(
            "/api/v2/product/get_item_list", 
            partner_id, partner_key, shop_id, access_token,
            {"offset": offset, "page_size": 100, "item_status": "NORMAL"}
        )
        
        if not data or "item" not in data:
            break
            
        item_ids.extend([item["item_id"] for item in data["item"]])
        has_next = data.get("has_next_page", False)
        offset = data.get("next_offset", 0)
        
    print(f"Found {len(item_ids)} active items.")
    return item_ids

In [ ]:
def extract_bucket_a_catalog(item_ids, partner_id, partner_key, shop_id, access_token):
    """Bucket A: Static Basic Info & Extra Info."""
    print("Extracting Static Catalog Info...")
    catalog_data = []
    
    # Shopee allows batching up to 50 items for base_info
    for i in range(0, len(item_ids), 50):
        batch = item_ids[i:i+50]
        batch_str = ",".join(map(str, batch))
        
        # 1. Get Base Info
        base_data = make_shopee_request(
            "/api/v2/product/get_item_base_info", 
            partner_id, partner_key, shop_id, access_token,
            {"item_id_list": batch_str}
        )
        
        # 2. Get Extra Info (usually holds pre-order status, views, etc.)
        extra_data = make_shopee_request(
            "/api/v2/product/get_item_extra_info", 
            partner_id, partner_key, shop_id, access_token,
            {"item_id_list": batch_str}
        )
        
        if base_data and "item_list" in base_data:
            for item in base_data["item_list"]:
                catalog_data.append(item)
                
    save_to_files(catalog_data, "bucket_A_catalog")

def extract_bucket_b_models(item_ids, partner_id, partner_key, shop_id, access_token):
    print(f"Extracting Dynamic Stock & Pricing for {len(item_ids)} items...")
    model_data = []
    
    for item_id in tqdm(item_ids, desc="Fetching Models"):
        data = make_shopee_request(
            "/api/v2/product/get_model_list", 
            partner_id, partner_key, shop_id, access_token,
            {"item_id": item_id}
        )
        
        if data and "tier_variation" in data:
            item_model_summary = {
                "item_id": item_id,
                "variations": data.get("tier_variation", []),
                "models": data.get("model", [])
            }
            model_data.append(item_model_summary)
        time.sleep(0.1) 
            
    save_to_files(model_data, "bucket_B_models")


def extract_bucket_c_comments_and_diagnostics(item_ids, partner_id, partner_key, shop_id, access_token):
    """Bucket C: Comments, Violations, and Diagnostics."""
    print("Extracting Comments and Diagnostics...")
    all_comments = []
    all_diagnostics = []
    
    for item_id in item_ids:
        # 1. Fetch Comments
        cursor = ""
        has_more = True
        while has_more:
            comment_data = make_shopee_request(
                "/api/v2/product/get_comment", 
                partner_id, partner_key, shop_id, access_token,
                {"item_id": item_id, "cursor": cursor, "page_size": 100}
            )
            
            if not comment_data:
                break
                
            all_comments.extend(comment_data.get("item_comment_list", []))
            has_more = comment_data.get("more", False)
            cursor = comment_data.get("next_cursor", "")
            
        # 2. Fetch Diagnostics
        diag_data = make_shopee_request(
            "/api/v2/product/get_item_content_diagnosis_result", 
            partner_id, partner_key, shop_id, access_token,
            {"item_id_list": str(item_id)} # Takes a string list
        )
        
        if diag_data and "item_list" in diag_data:
            all_diagnostics.extend(diag_data["item_list"])
            
    save_to_files(all_comments, "bucket_C_comments")
    save_to_files(all_diagnostics, "bucket_C_diagnostics")

In [ ]:
active_item_ids = get_all_item_ids(partner_id, partner_key, shop_id, access_token)

if active_item_ids:
    extract_bucket_a_catalog(active_item_ids, partner_id, partner_key, shop_id, access_token)
    extract_bucket_b_models(active_item_ids, partner_id, partner_key, shop_id, access_token)
    extract_bucket_c_comments_and_diagnostics(active_item_ids, partner_id, partner_key, shop_id, access_token)
    
    print("Pipeline Complete! Check the 'data_exports' folder.")